In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/fnlopezsanz/TP_Lab_Met_Cuantitativos_Inversiones/refs/heads/main/tasas-de-interes-por-depositos-a-30-dias-de-plazo-en-entidades-financieras-20250101-20251231.csv"
pf = pd.read_csv(url)

In [10]:
print("Dimensiones (filas, columnas):", pf.shape)
print("\nTipos de dato:")
print(pf.dtypes)

print("\nPrimeras filas:")
display(pf.head())

print("\nÚltimas filas:")
display(pf.tail())

# Tenemos 240 filas y 2 columnas.
# "Fecha" en formato object --> cambiar a datetime

Dimensiones (filas, columnas): (240, 2)

Tipos de dato:
Fecha     object
Valor    float64
dtype: object

Primeras filas:


,Fecha,Valor
0,2025-01-02,31.42
1,2025-01-03,31.80
2,2025-01-06,31.59
3,2025-01-07,31.01
4,2025-01-08,30.32



Últimas filas:


,Fecha,Valor
235,2025-12-22,26.04
236,2025-12-23,27.44
237,2025-12-26,25.38
238,2025-12-29,25.29
239,2025-12-30,25.46


In [11]:
# Limpieza:

pf["Fecha"] = pd.to_datetime(pf["Fecha"])

print("Día de apertura anual")
print(pf.loc[0])

print("\nNulos por columna:")
print(pf.isnull().sum())

print("\nFilas duplicadas:", pf.duplicated().sum())
print("Fechas duplicadas:", pf["Fecha"].duplicated().sum())

print("\nEstadísticas descriptivas:")
print(pf["Valor"].describe().round(2))

# Corregimos el formato de "Fecha".
# No hay nulos ni duplicados.
# Podemos ver que la media es más alta que la mediana. El máximo está alejado de ambas, lo que sugiere un período de tasas altas que levanta el promedio.

Día de apertura anual
Fecha    2025-01-02 00:00:00
Valor                  31.42
Name: 0, dtype: object

Nulos por columna:
Fecha    0
Valor    0
dtype: int64

Filas duplicadas: 0
Fechas duplicadas: 0

Estadísticas descriptivas:
count    240.00
mean      34.38
std        7.79
min       24.24
25%       29.46
50%       31.78
75%       37.50
max       56.06
Name: Valor, dtype: float64


In [12]:
pf["Fecha"] = pd.to_datetime(pf["Fecha"])
pf["mes"] = pf["Fecha"].dt.to_period("M")

# Rendimiento de un plazo fijo a 30 días
pf["rend_30d"] = pf["Valor"] / 100 * 30 / 365

mensual = (pf.groupby("mes")
           .agg(tna_inicio=("Valor", "first"),
                tna_prom=("Valor", "mean"),
                desv_est=("Valor", "std"),
                rend_inicio=("rend_30d", "first"),   # Si suponemos que renueva el primer día hábil del mes
                rend_prom=("rend_30d", "mean"),      # sensibilidad: promedio del mes
                dias_habiles=("Valor", "count"))
           .reset_index())

# Código y sugerencia de IA (incluye columnas "rend_inicio" y "rend_prom")
# cumprod() recorre una columna y en cada fila devuelve el producto de todos los valores desde el anteriores
mensual["rend_acum"] = (1 + mensual["rend_inicio"]).cumprod() - 1
mensual["rend_acum_prom"] = (1 + mensual["rend_prom"]).cumprod() - 1

display(mensual.round(4))

# Podemos concluir que un plazo fijo en 2025 rindió, aproximadamente, un 39,1%.

,mes,tna_inicio,tna_prom,desv_est,rend_inicio,rend_prom,dias_habiles,rend_acum,rend_acum_prom
0,2025-01,31.42,30.7641,0.8397,0.0258,0.0253,22,0.0258,0.0253
1,2025-02,26.99,27.5770,0.5410,0.0222,0.0227,20,0.0486,0.0485
2,2025-03,26.65,28.5339,1.0787,0.0219,0.0235,18,0.0715,0.0731
3,2025-04,31.07,32.4179,1.9248,0.0255,0.0266,19,0.0989,0.1017
4,2025-05,32.11,32.3010,0.6427,0.0264,0.0265,20,0.1279,0.1310
5,2025-06,32.75,31.6942,0.6956,0.0269,0.0261,19,0.1583,0.1604
6,2025-07,32.60,31.6918,1.5842,0.0268,0.0260,22,0.1893,0.1906
7,2025-08,36.03,45.6005,6.8080,0.0296,0.0375,20,0.2245,0.2353
8,2025-09,54.57,46.7968,6.0573,0.0449,0.0385,22,0.2795,0.2828
9,2025-10,40.17,44.7336,4.5667,0.0330,0.0368,22,0.3217,0.3299


In [13]:
# Estadísticos por trimestre
pf["Trimestre"] = pf["Fecha"].dt.to_period("Q")
por_trimestre = pf.groupby("Trimestre")["Valor"].agg(["mean", "std", "min", "max"]).round(2)

display(por_trimestre)

# De esta tabla podemos sacar algunas conclusiones:
# Los trimestres 3 y 4 tienen un desvío mucho mayor que los dos primeros.
# Esto significa que, en esos períodos (3 y 4) la tasa sufrió mucha más inestabilidad y variación que en los anteriores.

# Sugerencia de IA:
# Además de separar por trimestres, separar por períodos elegidos arbitrariamente: (ene-jul), (ago-oct), (nov-dic).
# Esto debido al comportamiento de las tasas (viendo las desviaciones estandar en la tabla mensual)

pf["Regimen"] = pd.cut(pf["Fecha"].dt.month,
                         bins=[0, 7, 10, 12],
                         labels=["ene-jul", "ago-oct", "nov-dic"])
por_regimen = pf.groupby("Regimen", observed=True)["Valor"].agg(["count", "mean", "std", "min", "max"]).round(2)

display(por_regimen)

# De esta forma queda en evidencia un período de gran volatilidad (agosto - octubre)
# ¿Cual puede haber sido la causa? ¿Elecciones? (legislativas 2025)

,mean,std,min,max
Trimestre,,,,
2025Q1,29.03,1.61,26.61,32.28
2025Q2,32.14,1.25,29.66,36.87
2025Q3,41.23,8.70,29.03,56.06
2025Q4,34.59,8.86,24.24,54.44


,count,mean,std,min,max
Regimen,,,,,
ene-jul,140,30.74,2.08,26.61,36.87
ago-oct,64,45.71,5.82,36.03,56.06
nov-dic,36,28.39,3.36,24.24,35.94


In [14]:
# FALTA HACER GRÁFICOS